In [1]:
from glob import glob
import pandas as pd
import os
import soundfile as sf
from tqdm import tqdm
from multiprocess import Pool
from scipy.io import wavfile
import itertools
import io
import numpy as np
import json
import re
import zipfile
from pathlib import Path

def chunks(l, n):
    for i in range(0, len(l), n):
        yield (l[i: i + n], i // n)

def multiprocessing(strings, function, cores=6, returned=True):
    df_split = chunks(strings, len(strings) // cores)
    pool = Pool(cores)
    pooled = pool.map(function, df_split)
    pool.close()
    pool.join()

    if returned:
        return list(itertools.chain(*pooled))

/usr/lib/python3/dist-packages/scipy/__init__.py:146: UserWarning: A NumPy version >=1.17.3 and <1.25.0 is required for this version of SciPy (detected version 1.26.4
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"


In [2]:
from huggingface_hub import snapshot_download

snapshot_download(repo_id="saeedzou/PersianVox_NM", 
                  repo_type="dataset", local_dir="./PersianVox_NM")

/home/ubuntu/.local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Fetching 17 files: 100%|██████████| 17/17 [00:08<00:00,  1.96it/s]


'/home/ubuntu/PersianVox_NM'

In [3]:
files = glob('PersianVox_NM/*/*.parquet')
len(files)

15

In [17]:
def loop(files):

    os.environ['OMP_NUM_THREADS'] = '1'
    os.environ['OPENBLAS_NUM_THREADS'] = '1'
    
    files, _ = files

    data = []
    for f in tqdm(files):
        base = f.split('/')[0] + '_audio'
        f_new = f.replace('/', '-').replace('.parquet', '')
        os.makedirs(base, exist_ok=True)
        df = pd.read_parquet(f)
        for i in range(len(df)):
            t = df['text'].iloc[i].strip()
            if len(t) < 2:
                continue
            audio_filename = f'{f_new}_{i}.mp3'
            audio_filename = os.path.join(base, audio_filename)
            b = df['audio'].iloc[i]['bytes']
            audio_np, sr = sf.read(io.BytesIO(b))
            if audio_np.ndim > 1:
                audio_np = audio_np.mean(axis=1)
            if audio_np.shape[0] < 10000:
                continue
            sf.write(audio_filename, audio_np, sr)
            
            data.append({
                'audio_filename': audio_filename,
                'text': t,
                'speaker': f"{base}"
            })
        
    return data

In [18]:
# data = loop((files[:1], 0))

In [19]:
data = multiprocessing(files, loop, len(files))

100%|██████████| 1/1 [05:39<00:00, 339.50s/it]


In [20]:
from datasets import Dataset

dataset = Dataset.from_list(data)
dataset[0]

{'audio_filename': 'PersianVox_NM_audio/PersianVox_NM-data-train-00002-of-00015_0.mp3',
 'text': 'ما نیز از جا برخواسته و لباسمان را پوشیده و آماده خروج شدیم',
 'speaker': 'PersianVox_NM_audio'}

In [21]:
dataset.push_to_hub('malaysia-ai/Multilingual-TTS', 'PersianVox_NM')

Creating parquet from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 46.06ba/s]
Processing Files (0 / 0): |          |  0.00B /  0.00B            
Processing Files (0 / 1):  92%|█████████▏| 3.39MB / 3.66MB,   ???B/s  
Processing Files (1 / 1): 100%|██████████| 3.66MB / 3.66MB, 2.76MB/s  
Processing Files (1 / 1): 100%|██████████| 3.66MB / 3.66MB, 1.39MB/s  
New Data Upload: 100%|██████████| 3.66MB / 3.66MB, 1.39MB/s  
Uploading the dataset shards: 100%|██████████| 1/1 [00:00<00:00,  1.62 shards/s]


CommitInfo(commit_url='https://huggingface.co/datasets/malaysia-ai/Multilingual-TTS/commit/5f6def0b6862cf6791dcb9c905fb6bd94b620225', commit_message='Upload dataset', commit_description='', oid='5f6def0b6862cf6791dcb9c905fb6bd94b620225', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/malaysia-ai/Multilingual-TTS', endpoint='https://huggingface.co', repo_type='dataset', repo_id='malaysia-ai/Multilingual-TTS'), pr_revision=None, pr_num=None)

In [22]:
audio_files = [d['audio_filename'] for d in data]

with open('PersianVox_NM-audio.json', 'w') as fopen:
    json.dump(list(set(audio_files)), fopen)

In [25]:
# !zip -rq PersianVox_NM_audio.zip PersianVox_NM_audio

In [26]:
# !hf upload malaysia-ai/Multilingual-TTS PersianVox_NM_audio.zip --repo-type=dataset

In [5]:
# !zip -rq PersianVox_NM_audio_neucodec.zip PersianVox_NM_audio_neucodec

In [4]:
# !hf upload malaysia-ai/Multilingual-TTS PersianVox_NM_audio_neucodec.zip --repo-type=dataset